In [7]:
# exploratory data analysis

In [8]:
# importing packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import ipywidgets as widgets
from IPython.display import display

In [9]:
# reading in data and identifying numeric data and categorical/binary data
df = pd.read_csv("Train1.csv", encoding='latin1')

numeric_cols = ['HOSPNUM', 'RDELAY', 'AGE', 'RSBP', 'HOURLOCAL', 'MINLOCAL', 'ONDRUG', 
    'DMAJNCHD', 'DSIDED', 'DRSISCD', 'DRSHD', 'DRSUNKD', 'DPED', 'DALIVED', 'DDEADD',
    'FLASTD', 'FDEADC', 'FU1_RECD', 'FU2_DONE', 'FU1_COMP', 'TD', 'EXPDD', 'EXPD6', 'EXPD14',
]

categorical_cols = ['RCONSC', 'SEX','RSLEEP', 'RATRIAL', 'RCT', 
    'RVISINF', 'RHEP24', 'RASP3', 'RDEF1', 'RDEF2', 'RDEF3', 'RDEF4', 'RDEF5',
    'RDEF6', 'RDEF7', 'RDEF8', 'STYPE', 'DAYLOCAL', 'RXASP', 'RXHEP', 'DASP14', 
    'DASPLT', 'DLH14', 'DMH14', 'DHH14', 'DSCH', 'DIVH', 'DAP', 'DOAC', 'DGORM', 
    'DSTER', 'DCAA', 'DHAEMD', 'DCAREND', 'DTHROMB', 'DMAJNCH', 'DSIDE', 'DDIAGISC', 
    'DDIAGHA', 'DDIAGUN', 'DNOSTRK', 'DRSISC', 'DRSH', 'DRSUNK', 'DPE', 'DALIVE',
    'DPLACE', 'DDEAD', 'DDEADC', 'FDEAD', 'FRECOVER', 'FDENNIS', 'FPLACE',
    'FAP', 'FOAC', 'COUNTRY', 'CNTRYNUM', 'CMPLASP', 'CMPLHEP', 'ID', 'SET14D', 
    'ID14', 'OCCODE', 'DEAD1', 'DEAD2', 'DEAD3', 'DEAD4', 'DEAD5', 'DEAD6', 'DEAD7', 
    'DEAD8', 'H14', 'ISC14', 'NK14', 'STRK14', 'HTI14', 'PE14', 'DVT14', 'TRAN14', 'NCB14' 
]

/var/folders/zy/kq72dddn62n0l225t1mm0w9m0000gr/T/ipykernel_9745/1580187524.py:2: DtypeWarning: Columns (32) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("Train1.csv", encoding='latin1')


In [10]:
discrete_feat = []
continuous_feat = []
for feat in df.select_dtypes(exclude='O'):
    if df[feat].nunique() <= 10:
        discrete_feat.append(feat)
    else:
        continuous_feat.append(feat)

print('Number of Discrete Features:',len(discrete_feat))
print('Number of Continuous Features:',len(continuous_feat))    

Number of Discrete Features: 25
Number of Continuous Features: 25


In [11]:
# plotting histograms with density lines for numeric variables

def plot_num(col):
    if col in df.columns and df[col].dropna().nunique() > 1:
        plt.figure(figsize=(8, 4))
        sns.distplot(df[col].dropna(), hist_kws=dict(linewidth=1, edgecolor='k'), bins=20)
        plt.title(f'Density Plot for {col}')
        plt.xlabel(col)
        plt.tight_layout()
        plt.ylabel('Density')
        plt.show()

dropdown = widgets.Dropdown(options=numeric_cols, description='Variable:')
widgets.interact(plot_num, col=dropdown)

interactive(children=(Dropdown(description='Variable:', options=('HOSPNUM', 'RDELAY', 'AGE', 'RSBP', 'HOURLOCA…

<function __main__.plot_num(col)>

In [12]:
# plotting pie charts for categorial/binary variables

def plot_cat(col):
    plt.figure(figsize=(6, 6))
    df[col].value_counts(dropna=False).plot.pie(
        autopct='%1.1f%%',
        startangle=90,
        wedgeprops=dict(width=0.5)
    )
    plt.title(f'Pie Chart of {col}')
    plt.ylabel('')
    plt.tight_layout()
    plt.show()

dropdown = widgets.Dropdown(options=categorical_cols, description='Variable:')
widgets.interact(plot_cat, col=dropdown)

interactive(children=(Dropdown(description='Variable:', options=('RCONSC', 'SEX', 'RSLEEP', 'RATRIAL', 'RCT', …

<function __main__.plot_cat(col)>

In [13]:
# cleaning data and numerically coding categorial variables

df_encoded = df.copy()

encoding_maps = {}

for col in categorical_cols:
    if col in df_encoded.columns:
        df_encoded[col], mapping = pd.factorize(df_encoded[col], sort=True)
        encoding_maps[col] = dict(enumerate(mapping))
    else:
        print(f"Column '{col}' not found in DataFrame — skipping.")

df_clean = df_encoded

print(df_clean.head())

   ID  HOSPNUM  RDELAY  RCONSC  SEX  AGE  RSLEEP  RATRIAL  RCT  RVISINF  ...  \
0   0        1      17       0    1   69       1       -1    1        1  ...   
1   1        1      10       1    1   76       1       -1    1        0  ...   
2   2        1      24       1    1   23       0       -1    1        0  ...   
3   3        1       5       0    0   83       0       -1    0        0  ...   
4   4        1       8       0    0   64       1       -1    1        1  ...   

   DEAD8  H14  ISC14  NK14  STRK14  HTI14  PE14  DVT14  TRAN14  NCB14  
0      0    0      0     0       0      0     0      0       0      0  
1      0    0      0     0       0      0     0      0       0      0  
2      0    0      0     0       0      0     0      0       0      0  
3      0    0      0     0       0      0     0      0       0      0  
4      0    0      0     0       0      0     0      0       0      0  

[5 rows x 113 columns]


In [20]:
# data cleaning code to use at start of each model/analysis

# bring in data
train = pd.read_csv('/Users/22holleranm/bios635/Train1.csv', encoding='latin1')

# all numeric cols
numeric_cols = ['HOSPNUM', 'RDELAY', 'AGE', 'RSBP', 'HOURLOCAL', 'MINLOCAL', 'ONDRUG', 
    'DMAJNCHD', 'DSIDED', 'DRSISCD', 'DRSHD', 'DRSUNKD', 'DPED', 'DALIVED', 'DDEADD',
    'FLASTD', 'FDEADC', 'FU1_RECD', 'FU2_DONE', 'FU1_COMP', 'TD', 'EXPDD', 'EXPD6', 'EXPD14'
]

# all cat calls
categorical_cols = ['RCONSC', 'SEX','RSLEEP', 'RATRIAL', 'RCT', 
    'RVISINF', 'RHEP24', 'RASP3', 'RDEF1', 'RDEF2', 'RDEF3', 'RDEF4', 'RDEF5',
    'RDEF6', 'RDEF7', 'RDEF8', 'STYPE', 'DAYLOCAL', 'RXASP', 'RXHEP', 'DASP14', 
    'DASPLT', 'DLH14', 'DMH14', 'DHH14', 'DSCH', 'DIVH', 'DAP', 'DOAC', 'DGORM', 
    'DSTER', 'DCAA', 'DHAEMD', 'DCAREND', 'DTHROMB', 'DMAJNCH', 'DSIDE', 'DDIAGISC', 
    'DDIAGHA', 'DDIAGUN', 'DNOSTRK', 'DRSISC', 'DRSH', 'DRSUNK', 'DPE', 'DALIVE',
    'DPLACE', 'DDEAD', 'DDEADC', 'FDEAD', 'FRECOVER', 'FDENNIS', 'FPLACE',
    'FAP', 'FOAC', 'COUNTRY', 'CNTRYNUM', 'CMPLASP', 'CMPLHEP', 'ID', 'SET14D', 
    'ID14', 'OCCODE', 'DEAD1', 'DEAD2', 'DEAD3', 'DEAD4', 'DEAD5', 'DEAD6', 'DEAD7', 
    'DEAD8', 'H14', 'ISC14', 'NK14', 'STRK14', 'HTI14', 'PE14', 'DVT14', 'TRAN14', 'NCB14' 
]

# encoding cat variables to be numeric
train_encoded = train.copy()

encoding_maps = {}

for col in categorical_cols:
    if col in train_encoded.columns:
        train_encoded[col], mapping = pd.factorize(train_encoded[col], sort=True)
        encoding_maps[col] = dict(enumerate(mapping))
    else:
        print(f"Column '{col}' not found in DataFrame — skipping.")

train_clean1 = train_encoded


# dropping unnecessary columns
train_clean2 = train_clean1.drop(['DDEAD', 'OCCODE', 'DDEADD', 'FDEAD', 'DDEADX', 'FDEADD', 'DDEADC', 'FDEADX', 'FDEADC', 'FLASTD', 'NCCODE', 'RDATE', 'DMAJNCHX', 'DSIDEX', 'DNOSTRKX'], axis =1)

# Fill columns with median/-1 for NaN values
for col in train_clean2.columns:
    if col in numeric_cols:
        train_clean2[col].fillna(train_clean2[col].median(), inplace=True)
    else:
        train_clean2[col].fillna(-1, inplace=True)


train_clean = train_clean2.copy()

train_clean.head()

# FDEAD IS ENCODED AS N = 0, Y = 1 (there are no U's)

/var/folders/zy/kq72dddn62n0l225t1mm0w9m0000gr/T/ipykernel_9745/219195626.py:4: DtypeWarning: Columns (32) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv('/Users/22holleranm/bios635/Train1.csv', encoding='latin1')


,ID,HOSPNUM,RDELAY,RCONSC,SEX,AGE,RSLEEP,RATRIAL,RCT,RVISINF,...,DEAD8,H14,ISC14,NK14,STRK14,HTI14,PE14,DVT14,TRAN14,NCB14
0,0,1,17,0,1,69,1,-1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,1,1,10,1,1,76,1,-1,1,0,...,0,0,0,0,0,0,0,0,0,0
2,2,1,24,1,1,23,0,-1,1,0,...,0,0,0,0,0,0,0,0,0,0
3,3,1,5,0,0,83,0,-1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,4,1,8,0,0,64,1,-1,1,1,...,0,0,0,0,0,0,0,0,0,0


In [22]:
# creating correlation matrix for numeric variables

train_clean.corr()['DIED'].sort_values(ascending=True).head(5)


FAP        -0.796698
TD         -0.796029
FDENNIS    -0.696673
FOAC       -0.675572
FRECOVER   -0.628414
Name: DIED, dtype: float64